# Band Structure for Twisted Molybdenum Disulfide bilayers at various angles.

## 0. Introduction.

This notebook demonstrates how to generate a twisted interface between two materials using commensurate lattices. The example uses molybdenum disulfide (MoS2) as both the film and substrate materials. The notebook uses the new `create_commensurate_interface` function which first creates a slab from the material and then performs commensurate lattice matching to find valid supercells for the target twist angle. The algorithm searches for supercell matrices within specified size limits to achieve the target twist angle within tolerance. The generated interface is visualized and analyzed to determine the actual twist angle and the number of atoms in the interface.
Then we calculate the band structure of the interface using a workflow, submitting a job and visualizing the results.

> **Kaihui Liu, Liming Zhang, Ting Cao, Chenhao Jin, Diana Qiu, Qin Zhou, Alex Zettl, Peidong Yang, Steve G. Louie & Feng Wang**
> Evolution of interlayer coupling in twisted molybdenum disulfide bilayers. Nature Communications, 5, 4966. 2014.
 > [https://doi.org/10.1038/ncomms5966](https://doi.org/10.1038/ncomms5966)

The twisted MoS2 bilayers are shown in the following Figure 4 from the article.

<img src="https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/interfaces/twisted-bilayer-molybdenum-disulfide/MoS2-twisted-bilayers.png" alt="Twisted MoS2 bilayers" width="600"/>

## 1. Prepare the Environment
### 1.1. Set up the notebook
Let's set angles and corresponding distances for the twisted interface from the article.

In [ ]:
# Uncomment lines to reproduce specific cases from the article
INTERFACE_PARAMETERS = [
    # {"angle": 0.0, "distance": 6.8},
    # {"angle": 13.0, "distance": 6.5},
    # {"angle": 22.0, "distance": 6.5},
    # {"angle": 38.0, "distance": 6.5},
    # {"angle": 47.0, "distance": 6.5},
    {"angle": 60.0, "distance": 6.1},  # AB1
]

# Slab creation parameters
MILLER_INDICES = (0, 0, 1)  # Miller indices for slab creation
NUMBER_OF_LAYERS = 1  # Number of layers in the slab

INTERFACE_VACUUM = 20.0  # in Angstroms

# Search algorithm parameters
MAX_REPETITION = None  # Maximum supercell matrix element value (None for automatic)
ANGLE_TOLERANCE = 0.5  # in degrees
RETURN_FIRST_MATCH = True  # If True, returns first solution within tolerance

# Visualization parameters
SHOW_INTERMEDIATE_STEPS = True
VISUALIZE_REPETITIONS = [3, 3, 1]

 ### 1.2. Install packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")

### 1.2a. Authenticate

Authenticate in the browser (OIDC device flow) or via JupyterLite host injection.

### 1.2b. Initialize API client

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
import os

from mat3ra.api_client import APIClient

client = APIClient.authenticate()
selected_account = client.my_account
OWNER_ID = os.getenv("ORGANIZATION_ID") or selected_account.id

### 1.3. Get input material
We'll use the MoS2 material from Standata.


In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

material = Material.create(Materials.get_by_name_and_categories("MoS2", "2D"))

print("Initial material properties:")
print(f"Formula: {material.formula}")
print(f"Number of atoms: {len(material.basis.elements.ids)}")

if SHOW_INTERMEDIATE_STEPS:
    visualize_materials(material, repetitions=VISUALIZE_REPETITIONS)
    visualize_materials(material, repetitions=VISUALIZE_REPETITIONS, rotation="-90x")

 ## 3. Generate Twisted Interface
 ### 3.1. Create slab


In [ ]:
from mat3ra.made.tools.modify import translate_to_z_level
from mat3ra.esse.models.core.reusable.axis_enum import AxisEnum
from mat3ra.made.tools.helpers import create_slab
from mat3ra.made.tools.helpers import create_interface_commensurate as create_commensurate_interface

slab = create_slab(
    crystal=material,
    miller_indices=MILLER_INDICES,
    number_of_layers=NUMBER_OF_LAYERS,
    vacuum=0.0,  # No vacuum in the slab, it is a 2D material
)
slab = translate_to_z_level(slab, "center")

visualize_materials(slab, rotation="-90x")

### 3.2. Create twisted interfaces

In [ ]:
interfaces = []
for parameters in INTERFACE_PARAMETERS:
    interface = create_commensurate_interface(
        material=slab,
        target_angle=parameters["angle"],
        angle_tolerance=ANGLE_TOLERANCE,
        max_repetition_int=MAX_REPETITION,
        return_first_match=RETURN_FIRST_MATCH,
        direction=AxisEnum.z,
        gap=parameters["distance"],
        vacuum=INTERFACE_VACUUM,
    )

    # resetting labels to not interfere with pseudopotential element names
    original_labels = interface.basis.labels
    interface.basis.set_labels_from_list([])
    interface.name = f"{interface.name} {parameters['angle']} degrees"
    interfaces.append(interface)
    print(f"Created interface with twist angle {parameters['angle']}° and {len(interface.basis.elements.ids)} atoms")

## 4. Preview the  materials


In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

for interface in interfaces:
    visualize_materials(interface, title=interface.name, viewer="wave")

## 5. Save materials

In [ ]:
from mat3ra.notebooks_utils.material import set_materials

set_materials(interfaces)

### 6.2. Add materials to your Collection via API


In [ ]:
from mat3ra.notebooks_utils.ui import display_JSON

OWNER_ID = os.getenv("ORGANIZATION_ID") or selected_account.id

saved_materials = []
for interface in interfaces:
    RAW_CONFIG = interface.to_dict()
    fields = ["name", "lattice", "basis"]
    CONFIG = {key: RAW_CONFIG[key] for key in fields}

    saved_material = client.materials.create(CONFIG, OWNER_ID)
    saved_materials.append(saved_material)

### 6.3. Verify material creation

In [ ]:
for saved_material in saved_materials:
    display_JSON(saved_material)

### 6.4. Get material id

In [ ]:
material_ids = [saved_material.get("_id") for saved_material in saved_materials]
print("Material IDs:", material_ids)

## 7. Setup workflow

### 7.1. Set Parameters

- **QUERY**: A query describing the documents to find. See [Meteor collection](https://docs.meteor.com/api/collections.html#Mongo-Collection-find) for more information.

- **limit**: Maximum number of results to return. See [Meteor collection](https://docs.meteor.com/api/collections.html#Mongo-Collection-find) for more information.

In [ ]:
OWNER_ID = os.getenv("ORGANIZATION_ID") or selected_account.id
WORKFLOW_SYSTEM_NAME = "espresso-band-structure"  # Name of the workflow to copy
RELAXATION_WORKFLOW_QUERY = {"systemName": "espresso-variable-cell-relaxation"}

OPTIONS = {"limit": 1}

### 7.2. Copy the workflow from the Bank to your collection


In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import copy_bank_workflow_by_system_name

band_structure_workflow_id = copy_bank_workflow_by_system_name(client.bank_workflows, WORKFLOW_SYSTEM_NAME, OWNER_ID)

### 7.3. Get the relaxation subworkflow

In [ ]:
relaxation_workflow = client.bank_workflows.list(RELAXATION_WORKFLOW_QUERY, OPTIONS)[0]
swf_0 = relaxation_workflow["subworkflows"][0]

### 7.4. Create the modifier for the workflow

Contact the endpoint to get the workflow by its ID. Adjust the necessary parameters in the workflow modifier to set the functional and method for the band gap calculation.

In [ ]:
QUERY = {"_id": band_structure_workflow_id, "owner._id": OWNER_ID}
workflow = client.workflows.list(QUERY, OPTIONS)[0]

modifier_0 = {
    "name": "Band Structure (LDA)",
}

swf_1 = workflow["subworkflows"][0]

model = {
    "type": "dft",
    "subtype": "lda",  # Is changed
    "method": {"type": "pseudopotential", "subtype": "us", "data": {}},
    "functional": {"slug": "pz"},  # Is changed
    "refiners": [],
    "modifiers": [],
}

### 7.5.Combine the subworkflows

In [ ]:
swf_0["model"] = model
swf_1["model"] = model

# remove the last unit from the subworkflow, since it is not needed for band structure calculation
del swf_1["units"][1]["next"]  # remove the connection to the last unit
swf_1["units"] = [swf_1["units"][0], swf_1["units"][1]]

modifier_1 = {"subworkflows": [swf_0, swf_1]}

### 7.6. Update workflow



In [ ]:
client.workflows.update(id_=band_structure_workflow_id, modifier=modifier_0)
client.workflows.update(id_=band_structure_workflow_id, modifier=modifier_1)

### 7.7. Get units

In [ ]:
unit_0 = relaxation_workflow["units"][0]
unit_1 = workflow["units"][0]

# Connect the units
unit_0["next"] = unit_1["flowchartId"]
unit_1["head"] = False

### 7.8. Create modifier for units

In [ ]:
modifier_2 = {"units": [unit_0, unit_1]}
client.workflows.update(id_=band_structure_workflow_id, modifier=modifier_2)

### 7.9. Verify the workflow

In [ ]:
# Fetch the updated workflow
workflow = client.workflows.list(QUERY, OPTIONS)[0]
display_JSON(workflow)

## 8. Create a job with the workflow and material
### 8.1. Set Parameters

In [ ]:
from datetime import datetime

OWNER_ID = os.getenv("ORGANIZATION_ID") or selected_account.id

project_id = client.projects.list({"isDefault": True, "owner._id": OWNER_ID})[0]["_id"]

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")  # for name to be found easily

JOB_NAME = f"Band Structure {timestamp}"

### 8.2. Create jobs

In [ ]:
jobs = client.jobs.create_by_ids(
    materials=saved_materials, workflow_id=workflow["_id"], project_id=project_id, prefix=JOB_NAME, owner_id=OWNER_ID
)

display_JSON(jobs)

### 8.3. Move jobs to set

In [ ]:
project_id = client.projects.list({"isDefault": True, "owner._id": OWNER_ID})[0]["_id"]

JOBS_SET_NAME = None

if JOBS_SET_NAME is not None:
    jobs_set = client.jobs.create_set({"name": JOBS_SET_NAME, "projectId": project_id, "owner": {"_id": OWNER_ID}})
    for job in jobs:
        client.jobs.move_to_set(job["_id"], "", jobs_set["_id"])

workflow_in_job = jobs[0]["workflow"]

### 8.4. Update job with new parameters

In [ ]:
kgrid_relax = [6, 6, 1]

modifier_relax = {
    "workflow.subworkflows.0.units.0.context": {
        "kgrid": {
            "dimensions": kgrid_relax,
            "shifts": [0, 0, 0],
            "reciprocalVectorRatios": [1, 1, 1],
            "gridMetricType": "KPPRA",
            "gridMetricValue": 2,
            "preferGridMetric": False,
        },
        "isKgridEdited": True,
        "subworkflowContext": {},
    }
}

### 8.5. Update each job with the modifier

In [ ]:
for job in jobs:
    client.jobs.update(id_=job["_id"], modifier={"updateParameters": {"skipRender": False}, **modifier_relax})

## 9. Submit the job

In [ ]:
for job in jobs:
    client.jobs.submit(job["_id"])

## 10. Get results
### 10.1. Wait for jobs to finish

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

job_ids = [job["_id"] for job in jobs]

await wait_for_jobs_to_finish_async(client.jobs, job_ids, poll_interval=60)

### 10.2. Get the Band Gap property

In [ ]:
results = []
for material in saved_materials:
    job = next((job for job in jobs if job["_material"]["_id"] == material["_id"]))
    unit_flowchart_id_0 = job["workflow"]["subworkflows"][1]["units"][0]["flowchartId"]
    fermi_level = client.properties.get_property(
        job["_id"],
        unit_flowchart_id=unit_flowchart_id_0,
        property_name="fermi_energy",
    )

    unit_flowchart_id_1 = job["workflow"]["subworkflows"][1]["units"][1]["flowchartId"]
    band_structure = client.properties.get_property(
        job["_id"],
        unit_flowchart_id=unit_flowchart_id_1,
        property_name="band_structure",
    )

    results.append(
        {
            "material_id": material["_id"],
            "fermi_level": fermi_level,
            "band_structure": band_structure,
        }
    )

### 10.3. Plotting functions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def get_high_symmetry_points():
    """High-symmetry points for 2D hexagonal lattices (like MoS2)"""
    return {
        (0, 0, 0): "Γ",
        (0.5, 0, 0): "M",
        (0, 0.5, 0): "M",
        (-0.5, 0, 0): "M",
        (0, -0.5, 0): "M",
        (1 / 3, 1 / 3, 0): "K",
        (-1 / 3, 1 / 3, 0): "K",
        (1 / 3, -1 / 3, 0): "K",
        (-1 / 3, -1 / 3, 0): "K",
    }


def find_high_symmetry_labels(xDataArray, tolerance=0.05):
    """Find high-symmetry points in the k-path"""
    high_sym_points = get_high_symmetry_points()
    labels = []
    positions = []

    for i, k_point in enumerate(xDataArray):
        k_frac = np.array(k_point)
        best_match = None
        best_distance = float("inf")

        for sym_k, label in high_sym_points.items():
            distance = np.linalg.norm(k_frac - np.array(sym_k))
            if distance < tolerance and distance < best_distance:
                best_distance = distance
                best_match = (label, i)

        if best_match:
            label, pos = best_match
            # Avoid consecutive duplicates
            if not labels or labels[-1] != label:
                labels.append(label)
                positions.append(pos)

    return positions, labels


def plot_band_structure_with_labels(band_structure_data, ylim=None, title="Band Structure"):
    xDataArray = band_structure_data["xDataArray"]
    yDataSeries = band_structure_data["yDataSeries"]

    # Calculate k-point distances
    k_distances = [0]
    for i in range(1, len(xDataArray)):
        prev_k = np.array(xDataArray[i - 1])
        curr_k = np.array(xDataArray[i])
        distance = np.linalg.norm(curr_k - prev_k)
        k_distances.append(k_distances[-1] + distance)

    # Find high-symmetry points
    sym_positions, sym_labels = find_high_symmetry_labels(xDataArray)

    # Plot bands
    plt.figure(figsize=(10, 6))
    for energies in yDataSeries:
        plt.plot(k_distances, energies, "b-", linewidth=1)

    # Add vertical lines at high-symmetry points
    for pos in sym_positions:
        plt.axvline(x=k_distances[pos], color="r", linestyle="--", alpha=0.5)

    # Set x-axis labels
    if sym_positions and sym_labels:
        plt.xticks([k_distances[pos] for pos in sym_positions], sym_labels)

    # Set y-axis limits
    if ylim:
        plt.ylim(ylim)

    plt.xlabel("k-point path")
    plt.ylabel("Energy (eV)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 10.4. Plot the band structure

In [ ]:
MAX_E = 4.0  # Maximum energy for y-axis limit, eV
MIN_E = -2.0  # Minimum energy for y-axis limit, eV

for result in results:
    if result["band_structure"]:
        band_data = result["band_structure"]["data"]
        # adjust for Fermi level
        fermi_level = result["fermi_level"]["data"]["value"]
        for i in range(len(band_data["yDataSeries"])):
            band_data["yDataSeries"][i] = [e - fermi_level for e in band_data["yDataSeries"][i]]

        plot_band_structure_with_labels(band_data, ylim=[MIN_E, MAX_E])